#### Embeddings - представление слов в виде вектора чисел, отражающее смысл

king - man + woman = queen

paris - france + italy = rome


In [3]:
import gensim # Библиотека для работы с  Word2Vec, Doc2Vec и с другими моделями
import numpy as np
from gensim import downloader as api
from gensim.models import Word2Vec

import numpy as np

import matplotlib.pyplot as plt
from pyexpat import model

from  sklearn.manifold import TSNE

print(gensim.__version__)

4.4.0


## 1. Word Embeddings

#### Word Embeddings

In [5]:
vocab = ["кот", "собака", "птица", "рыба", "слон"]

print("One Hot encoding:")

print("-"*50)

for i, word in enumerate(vocab):
    one_hot = np.zeros(len(vocab), dtype=int)
    one_hot[i] = 1
    print(f"{word:10} = {one_hot}")

print("-")
print("Dense embedings:")
print("-"*50)

embeddings = {
    "кот": np.array([0.8, 0.2, -0.1, 0.5]), # Домашние животные
    "собака": np.array([0.7, 0.3, -0.2, 0.5]), # похожи
    "птица": np.array([0.1, 0.9, 0.8, -0.1]), # летающие - другие
    "рыба": np.array([-0.5, 0.1, 0.3, 0.9]), # водные - совсем другие
    "слон": np.array([0.6, 0.4, -0.3, 0.4]), # крупное животное
}

for word, vec in embeddings.items():
    print(f"{word:10} = {vec}")


One Hot encoding:
--------------------------------------------------
кот        = [1 0 0 0 0]
собака     = [0 1 0 0 0]
птица      = [0 0 1 0 0]
рыба       = [0 0 0 1 0]
слон       = [0 0 0 0 1]
-
Dense embedings:
--------------------------------------------------
кот        = [ 0.8  0.2 -0.1  0.5]
собака     = [ 0.7  0.3 -0.2  0.5]
птица      = [ 0.1  0.9  0.8 -0.1]
рыба       = [-0.5  0.1  0.3  0.9]
слон       = [ 0.6  0.4 -0.3  0.4]


In [6]:
# Вычисление косинусного сходства между векторами
def cosine_similarity(vec1, vec2):
    """
        cos(θ) = (A · B) / (||A|| × ||B||)
    """
    dot_product = np.dot(vec1, vec2)

    norm1 = np.linalg.norm(vec1) # ||A|| = sqrt(a1² + a2² + ...)
    norm2 = np.linalg.norm(vec2) # ||A|| = sqrt(a1² + a2² + ...)

    return dot_product / (norm1 * norm2)

print("Cosine similarity:")
print("-"*50)

words = list(embeddings.keys())
for i in range(len(words)):
    for j in range(i+1, len(words)):
        word1, word2 = words[i], words[j]
        sim = cosine_similarity(embeddings[word1], embeddings[word2])
        print(f"{word1:10} - {word2:10} = {sim}")

Cosine similarity:
--------------------------------------------------
кот        - собака     = 0.9841616856627443
кот        - птица      = 0.11059124777543396
кот        - рыба       = 0.03830602342540456
кот        - слон       = 0.9285767423913251
собака     - птица      = 0.1149542569667464
собака     - рыба       = 0.06968020490219626
собака     - слон       = 0.9774284999977502
птица      - рыба       = 0.14550098687084356
птица      - слон       = 0.13159033899195383
рыба       - слон       = 0.010580973892262188


## 2. Word2Vec: Обучение собственной модели

In [9]:
corpus = [
    ["кот", "сидит", "на", "диване", "и", "спит"],
    ["собака", "лежит", "на", "ковре", "и", "отдыхает"],
    ["кот", "ловит", "мышь", "в", "саду"],
    ["собака", "бегает", "по", "улице"],
    ["птица", "летит", "в", "небе"],
    ["рыба", "плавает", "в", "воде"],
    ["кот", "и", "собака", "играют", "вместе"],
    ["птица", "сидит", "на", "дереве"],
    ["рыба", "живёт", "в", "аквариуме"],
    ["собака", "охраняет", "дом"],
    ["кот", "мурлычет", "от", "удовольствия"],
    ["большая", "собака", "лает", "громко"],
    ["маленький", "кот", "спит", "в", "корзине"],
    ["красивая", "птица", "поёт", "песню"],
    ["золотая", "рыба", "плавает", "в", "пруду"]
]

print(f"{len(corpus)} sentences in corpus")
print(f"Sentences example: {corpus[0]}")

all_words = set()
for sentence in corpus:
    all_words.update(sentence)
print(f"{len(all_words)} unique words in corpus")


15 sentences in corpus
Sentences example: ['кот', 'сидит', 'на', 'диване', 'и', 'спит']
43 unique words in corpus


In [10]:
model = Word2Vec(
    sentences=corpus,
    vector_size=50,
    window=5,
    min_count=1,
    workers=4,
    epochs=100,
    sg=1, # 1 Skip-gram (контекст по слову), 0 = CBOW(слова по контексту)
)

print(len(model.wv))
print(model.wv.vector_size)
print(list(model.wv.key_to_index.keys()))

43
50
['в', 'собака', 'кот', 'рыба', 'птица', 'и', 'на', 'плавает', 'спит', 'сидит', 'пруду', 'золотая', 'песню', 'поёт', 'красивая', 'корзине', 'маленький', 'громко', 'лает', 'большая', 'удовольствия', 'от', 'мурлычет', 'дом', 'охраняет', 'аквариуме', 'живёт', 'дереве', 'вместе', 'играют', 'воде', 'небе', 'летит', 'улице', 'по', 'бегает', 'саду', 'мышь', 'ловит', 'отдыхает', 'ковре', 'лежит', 'диване']


In [13]:
cat_vector = model.wv["кот"]
print(cat_vector[:10])

dog_vector = model.wv["собака"]
print(dog_vector[:10])

[-0.01776348  0.00801032  0.01071676  0.0117433   0.01494474 -0.01335139
  0.00344423  0.01468321 -0.00647001 -0.01393805]
[-0.0169459   0.00981907 -0.00802178  0.00175042  0.01703472 -0.00965092
  0.010006   -0.0120267  -0.0079057   0.01758802]


In [16]:
similar_to_cat = model.wv.most_similar("кот", topn=5)

for word, similarity in similar_to_cat:
    print(f"{word:15} - {similarity:.4f}")

print("-"*50)

similar_to_dog = model.wv.most_similar("собака", topn=5)

for word, similarity in similar_to_dog:
    print(f"{word:15} - {similarity:.4f}")

саду            - 0.2637
бегает          - 0.2616
птица           - 0.2182
в               - 0.2158
лает            - 0.2130
--------------------------------------------------
воде            - 0.3800
вместе          - 0.2499
маленький       - 0.2377
аквариуме       - 0.2337
улице           - 0.1917


In [20]:
similar_cat_dog = model.wv.similarity("кот", "собака")
print(f"Similarity words: {similar_cat_dog:4f}")

similar_cat_fish = model.wv.similarity("кот", "рыба")
print(f"Similarity words: {similar_cat_fish:4f}")

similar_bird_fly = model.wv.similarity("птица", "летит")
print(f"Similarity words: {similar_bird_fly:4f}")


similar_fish_swim = model.wv.similarity("рыба", "плавает")
print(f"Similarity words: {similar_fish_swim:4f}")


Similarity words: 0.066631
Similarity words: -0.131792
Similarity words: -0.086428
Similarity words: 0.058461


In [ ]:
# Арифметика с векторами слов
print()